In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
import urllib.request
from collections import deque
import threading
import random
import time

model_path = "pose_landmarker.task"
urllib.request.urlretrieve(
    "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task",
    model_path
)

latest_landmarks = []
landmarks_lock = threading.Lock()

def result_callback(result, output_image, timestamp_ms):
    global latest_landmarks
    with landmarks_lock:
        latest_landmarks = result.pose_landmarks if result.pose_landmarks else []

base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.PoseLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.LIVE_STREAM,
    min_pose_detection_confidence=0.4,
    min_pose_presence_confidence=0.4,
    min_tracking_confidence=0.5,
    num_poses=1,
    result_callback=result_callback
)
landmarker = vision.PoseLandmarker.create_from_options(options)

torso_history = deque(maxlen=5)

CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),(11,12),
    (11,13),(13,15),(12,14),(14,16),(15,17),(15,19),(17,19),
    (16,18),(16,20),(18,20),(11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(27,31),(24,26),(26,28),(28,30),(28,32),
]

CYAN        = (255, 255, 0)
NEON_GREEN  = (0, 255, 128)
NEON_PINK   = (180, 0, 255)
NEON_ORANGE = (0, 165, 255)
WHITE       = (255, 255, 255)
BLACK       = (0, 0, 0)

player_w, player_h   = 60, 80
obstacle_w, obstacle_h = 50, 30
obstacle_speed  = 6
spawn_interval  = 1.5

obstacles       = []
score           = 0
lives           = 3
last_spawn_time = time.time()
hit_flash_until = 0
game_over       = False


def glow_rect(frame, x1, y1, x2, y2, color, thickness=2, glow=6):
    for i in range(glow, 0, -2):
        ov = frame.copy()
        cv2.rectangle(ov, (x1-i, y1-i), (x2+i, y2+i), color, thickness+i)
        cv2.addWeighted(ov, 0.15, frame, 0.85, 0, frame)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)

def glow_line(frame, p1, p2, color, thickness=1, glow=3):
    for i in range(glow, 0, -1):
        ov = frame.copy()
        cv2.line(ov, p1, p2, color, thickness+i)
        cv2.addWeighted(ov, 0.12, frame, 0.88, 0, frame)
    cv2.line(frame, p1, p2, color, thickness)

def glow_circle(frame, center, radius, color):
    for i in range(4, 0, -2):
        ov = frame.copy()
        cv2.circle(ov, center, radius+i, color, 2)
        cv2.addWeighted(ov, 0.12, frame, 0.88, 0, frame)
    cv2.circle(frame, center, radius, color, -1)

def draw_scanlines(frame):
    ov = np.zeros_like(frame)
    for y in range(0, frame.shape[0], 4):
        cv2.line(ov, (0, y), (frame.shape[1], y), BLACK, 1)
    cv2.addWeighted(ov, 0.08, frame, 1.0, 0, frame)

def draw_hud(frame, w, h, score, lives):
    cv2.rectangle(frame, (0, 0), (w, 55), (10, 5, 20), -1)
    cv2.line(frame, (0, 55), (w, 55), NEON_PINK, 1)
    cv2.putText(frame, "SCORE", (20, 20), cv2.FONT_HERSHEY_DUPLEX, 0.5, NEON_PINK, 1)
    cv2.putText(frame, str(score), (20, 48), cv2.FONT_HERSHEY_DUPLEX, 1.0, WHITE, 2)
    cv2.putText(frame, "LIVES", (w-140, 20), cv2.FONT_HERSHEY_DUPLEX, 0.5, NEON_PINK, 1)
    for i in range(3):
        glow_circle(frame, (w-120+i*35, 40), 10, NEON_GREEN if i < lives else (50, 50, 50))
    cv2.putText(frame, "DODGE", (w//2-55, 38), cv2.FONT_HERSHEY_DUPLEX, 1.0, CYAN, 2)


cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
cap.set(cv2.CAP_PROP_FPS, 30)

timestamp_ms    = 0
smooth_torso_cx = 300

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame = cv2.flip(frame, 1)
    h, w = frame.shape[:2]

    cv2.addWeighted(frame, 0.45, np.zeros_like(frame), 0.55, 0, frame)

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                        data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    landmarker.detect_async(mp_image, timestamp_ms)
    timestamp_ms += 1

    with landmarks_lock:
        current_landmarks = list(latest_landmarks)
    if current_landmarks:
        lm  = current_landmarks[0]
        pts = [(int(l.x * w), int(l.y * h)) for l in lm]
        for a, b in CONNECTIONS:
            if lm[a].visibility > 0.5 and lm[b].visibility > 0.5:
                glow_line(frame, pts[a], pts[b], NEON_GREEN)
        for i, (px, py) in enumerate(pts):
            if lm[i].visibility > 0.5:
                glow_circle(frame, (px, py), 3, NEON_GREEN)
        raw_cx = (((lm[11].x+lm[12].x)/2 + (lm[23].x+lm[24].x)/2) / 2) * w
        torso_history.append(raw_cx)
        smooth_torso_cx = int(np.mean(torso_history))

    player_cx = int(np.clip(smooth_torso_cx, player_w//2, w-player_w//2))
    px1, px2  = player_cx - player_w//2, player_cx + player_w//2
    py1, py2  = h - player_h - 10, h - 10

    if not game_over:
        now = time.time()
        if now - last_spawn_time > spawn_interval:
            ox = random.randint(0, w - obstacle_w)
            obstacles.append([ox, 0, ox+obstacle_w, obstacle_h,
                               random.choice([NEON_PINK, NEON_ORANGE, CYAN])])
            last_spawn_time = now

        surviving = []
        for obs in obstacles:
            obs[1] += obstacle_speed
            obs[3] += obstacle_speed
            if not (obs[2] < px1 or obs[0] > px2 or obs[3] < py1 or obs[1] > py2):
                lives -= 1
                hit_flash_until = now + 0.4
                if lives <= 0:
                    game_over = True
            elif obs[1] > h:
                score += 1
            else:
                surviving.append(obs)
        obstacles = surviving

        for obs in obstacles:
            glow_rect(frame, obs[0], obs[1], obs[2], obs[3], obs[4], thickness=-1, glow=6)
            glow_rect(frame, obs[0], obs[1], obs[2], obs[3], WHITE,   thickness=1,  glow=2)

        p_color = NEON_PINK if now < hit_flash_until else CYAN
        glow_rect(frame, px1, py1, px2, py2, p_color, thickness=-1, glow=10)
        glow_rect(frame, px1, py1, px2, py2, WHITE,   thickness=1,  glow=3)
        glow_line(frame, (0, h-8), (w, h-8), CYAN)
        draw_hud(frame, w, h, score, lives)
        draw_scanlines(frame)

    else:
        ov = frame.copy()
        cv2.rectangle(ov, (0, 0), (w, h), BLACK, -1)
        cv2.addWeighted(ov, 0.75, frame, 0.25, 0, frame)
        pulse = int(abs(np.sin(time.time() * 3)) * 40)
        cv2.putText(frame, "GAME OVER", (w//2-175, h//2-50),
                    cv2.FONT_HERSHEY_DUPLEX, 2.2, (255, pulse, 180), 4)
        cv2.putText(frame, f"FINAL SCORE  {score}", (w//2-145, h//2+20),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, CYAN, 2)
        cv2.putText(frame, "[ R ]  RESTART      [ Q ]  QUIT", (w//2-185, h//2+75),
                    cv2.FONT_HERSHEY_DUPLEX, 0.7, NEON_GREEN, 1)
        draw_scanlines(frame)

    cv2.imshow("DODGE", frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('q'):
        break
    if key == ord('r') and game_over:
        obstacles.clear()
        score, lives, game_over = 0, 3, False
        last_spawn_time = time.time()

cap.release()
cv2.destroyAllWindows()